# MVP — Diagnóstico de calidad de red y soporte a decisión de inversión/mantenimiento

**TFM Grupo 1 — OBS Business School**  
**Ejecutar:** Kernel → Restart → Run All (debe completar sin error)

---

| Sección | Owner | Input | Output |
|---------|-------|-------|--------|
| §0 Setup | Dev-1 | — | Constantes, paths |
| §1 Carga | Dev-1 | `db_files/` | `df_claims`, `df_location`, `df_kpis` |
| §2 Dataset semanal | Dev-1 | DataFrames crudos | `df_weekly` + Parquet |
| §3 EDA | Dev-2 | `df_weekly` | Figuras, hallazgos |
| §4 Features + split | Dev-1 | `df_weekly` | `X_train/test`, `y_train/test` |
| §5 Modelos | Dev-2 | X/y splits | `model_rules/lr/xgb` |
| §6 Evaluación | Dev-2 | predicciones | `df_metrics`, figura AUC-PR |
| §7 SHAP | Dev-3 | `model_xgb`, `X_test` | `shap_values`, figuras |
| §8 Clustering | Dev-3 | `df_weekly` | `df_sites_clustered` |
| §9 Ranking | Dev-3 | score + SHAP + clusters | `df_ranking` + Parquet |

In [ ]:
# Colab no trae optuna/shap/xgboost preinstalados -- instalar solo si hace falta.
try:
    import google.colab  # noqa: F401
    %pip install -q optuna shap xgboost
except ImportError:
    pass

## §0 — Setup & configuración global

**Owner:** Dev-1  
**Input:** —  
**Output:** constantes globales, paths  
**Dependencias:** ninguna

In [ ]:
from __future__ import annotations
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Detecta Colab vs. entorno local — permite un solo notebook para ambos casos
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Carpeta dedicada del notebook en Drive (separada del remote DVC)
    drive.mount('/content/drive')
    ROOT_DIR = Path('/content/drive/MyDrive/TFM-local')
else:
    # Permite importar desde src/ independientemente del CWD
    ROOT_DIR = Path().resolve().parent

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

# — Reproducibilidad —
RANDOM_STATE      = 42

# — Parámetros de modelado —
ROLLING_WINDOW    = 7            # días
WEEK_ANCHOR       = "W-SUN"      # fin de semana = domingo
TEST_CUTOFF_DATE  = "2026-02-01" # split temporal train/test

# — Umbrales de negocio (derivados del EDA: entregables/eda-hallazgos.md) —
AVA_THRESHOLD     = 0.98         # tasa queja 3-4x por debajo
DC_THRESHOLD      = 0.02         # tasa queja 2x por encima
THP_THRESHOLD_MBP = 15.0         # Mbps, tasa queja 2-3x por debajo

# — Columnas clave —
SITE_ID_COL  = "site_id"
WEEK_COL     = "semana"
TARGET_COL   = "es_reclamo_semanal"

# — Paths —
OUTPUT_DIR  = ROOT_DIR / "output"
FIGURES_DIR = OUTPUT_DIR / "figures"
DATA_WEEKLY = OUTPUT_DIR / "modeling_dataset_weekly.parquet"
RANKING_PATH = OUTPUT_DIR / "ranking_sitios.parquet"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"ROOT_DIR   : {ROOT_DIR}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")
print(f"RANDOM_STATE = {RANDOM_STATE} | TEST_CUTOFF = {TEST_CUTOFF_DATE}")

## §1 — Carga y validación de datos

**Owner:** Dev-1  
**Input:** archivos en `db_files/`  
**Output:** `df_claims`, `df_location`, `df_kpis`  
**Dependencias:** §0

In [ ]:
# Si el Parquet semanal ya existe (workflow Colab compartido), no hace falta
# el repo completo ni db_files/ crudo — Dev-2/Dev-3 solo consumen df_weekly (§2).
if DATA_WEEKLY.exists():
    df_claims = df_location = df_kpis = None
    print("Parquet semanal ya disponible — se omite carga de datos crudos (§1).")
else:
    from src.build_modeling_dataset import load_from_files, normalize, validate_integrity

    df_claims, df_location, df_kpis = load_from_files(ROOT_DIR / "db_files")
    df_claims, df_location, df_kpis = normalize(df_claims, df_location, df_kpis)
    validate_integrity(df_claims, df_location, df_kpis)

    print(f"claims   : {df_claims.shape}")
    print(f"location : {df_location.shape}")
    print(f"kpis     : {df_kpis.shape}")

## §2 — Dataset sitio × semana

**Owner:** Dev-1  
**Input:** `df_claims`, `df_location`, `df_kpis`  
**Output:** `df_weekly` → `output/modeling_dataset_weekly.parquet`  
**Dependencias:** §1

Punto de handoff crítico (H1): Dev-2 y Dev-3 leen desde el Parquet.

In [ ]:
# TODO Dev-1:
# 1. Agregar claims a sitio × semana (target = n_reclamos_semana > 0)
# 2. KPIs rolling leakage-safe (.shift(1).rolling(ROLLING_WINDOW))
# 3. Normalizar por USERS_4G
# 4. Guardar Parquet

# df_weekly = ...build logic here...
# df_weekly.to_parquet(DATA_WEEKLY, index=False)

# --- Si el Parquet ya existe, cargar directamente ---
if DATA_WEEKLY.exists():
    df_weekly = pd.read_parquet(DATA_WEEKLY)
    print(f"df_weekly cargado desde Parquet: {df_weekly.shape}")
    print(f"Positivos: {df_weekly[TARGET_COL].sum()} / {len(df_weekly)} = {df_weekly[TARGET_COL].mean():.2%}")
else:
    raise FileNotFoundError(f"Parquet no encontrado: {DATA_WEEKLY}. Ejecutar build_modeling_dataset.py primero.")

## §3 — EDA diagnóstico

**Owner:** Dev-2  
**Input:** `df_weekly` (vía Parquet si §2 no está corrido)  
**Output:** figuras en `output/figures/`, hallazgos documentados  
**Dependencias:** §2 (Parquet H1)

In [ ]:
# Asegurar que df_weekly está disponible (puede cargar sin correr §2)
if "df_weekly" not in dir():
    df_weekly = pd.read_parquet(DATA_WEEKLY)

# TODO Dev-2:
# 1. Distribución del target (imbalance)
# 2. Tasa queja vs. bucket AVA (fig_ava_bucket)
# 3. Tasa queja vs. bucket DC (fig_dc_bucket)
# 4. Tasa queja vs. bucket THP (fig_thp_bucket)
# 5. Evolución temporal de quejas y KPIs (fig_temporal)
# 6. Correlación features vs. TARGET_COL (fig_correlation)
#
# Guardar figuras:
# fig.savefig(FIGURES_DIR / "eda_ava_bucket.png", dpi=150, bbox_inches="tight")

> **Hallazgo [completar]:** ...  
> **Implicación:** ...

## §4 — Feature engineering y split temporal

**Owner:** Dev-1  
**Input:** `df_weekly`  
**Output:** `X_train`, `X_test`, `y_train`, `y_test`, `FEATURE_COLS`  
**Dependencias:** §2

Punto de handoff H2: Dev-2 y Dev-3 consumen estas variables directamente.

In [ ]:
# TODO Dev-1: definir FEATURE_COLS con las columnas disponibles en df_weekly
FEATURE_COLS = [
    "avg_AVA_4G_7d",
    "max_DC_V4G_7d",
    "avg_THP_4G_7d",
    "degradacion_THP_7d",
    "avg_CSFR_V4G_7d",
    "USERS_4G",
    # "reclamos_por_usuario",  # agregar cuando esté en df_weekly
    # Categoricals one-hot-encoded
    # "tipo_negocio_B2C", "tipo_negocio_B2B", "tipo_negocio_PREPAGO",
]

# Split temporal — nunca aleatorio en series de tiempo
mask_train = df_weekly[WEEK_COL] < TEST_CUTOFF_DATE
df_train   = df_weekly[mask_train].copy()
df_test    = df_weekly[~mask_train].copy()

X_train = df_train[FEATURE_COLS]
y_train = df_train[TARGET_COL]
X_test  = df_test[FEATURE_COLS]
y_test  = df_test[TARGET_COL]

print(f"Train: {X_train.shape} | positivos={y_train.sum()} ({y_train.mean():.2%})")
print(f"Test : {X_test.shape}  | positivos={y_test.sum()} ({y_test.mean():.2%})")
print(f"Features ({len(FEATURE_COLS)}): {FEATURE_COLS}")

## §5 — Modelos (baseline / LR / XGBoost)

**Owner:** Dev-2  
**Input:** `X_train`, `X_test`, `y_train`, `y_test`, `FEATURE_COLS`, `RANDOM_STATE`  
**Output:** `model_rules`, `model_lr`, `model_xgb`, `y_pred_*`, `y_prob_*`  
**Dependencias:** §4 (handoff H2)

In [ ]:
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
import optuna

# --- Baseline de reglas ---
# TODO Dev-2: implementar como clase o función con .predict() y .predict_proba()
# Flag si AVA < AVA_THRESHOLD OR DC > DC_THRESHOLD OR THP < THP_THRESHOLD_MBP
# model_rules = ...
# y_pred_rules = model_rules.predict(X_test)
# y_prob_rules = model_rules.predict_proba(X_test)[:, 1]

# --- Logistic Regression ---
# model_lr = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)
# model_lr.fit(X_train, y_train)
# y_pred_lr = model_lr.predict(X_test)
# y_prob_lr = model_lr.predict_proba(X_test)[:, 1]

# --- XGBoost + Optuna ---
# scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
# TODO Dev-2: definir objetivo Optuna, max 50 trials, métrica average_precision
# model_xgb = XGBClassifier(..., random_state=RANDOM_STATE)
# model_xgb.fit(X_train, y_train)
# y_pred_xgb = model_xgb.predict(X_test)
# y_prob_xgb = model_xgb.predict_proba(X_test)[:, 1]

print("§5 pendiente — Dev-2")

## §6 — Evaluación comparativa

**Owner:** Dev-2  
**Input:** `y_test`, `y_pred_*`, `y_prob_*`  
**Output:** `df_metrics`, `fig_aucpr`  
**Dependencias:** §5

In [ ]:
from sklearn.metrics import (
    average_precision_score, precision_recall_curve,
    recall_score, precision_score, f1_score
)

# TODO Dev-2: calcular métricas para los 3 modelos
# df_metrics = pd.DataFrame([
#     {"modelo": "Reglas", "AUC-PR": ..., "Recall": ..., "Precision": ..., "F1": ..., "n_alertas": ...},
#     {"modelo": "LR",     "AUC-PR": ..., "Recall": ..., "Precision": ..., "F1": ..., "n_alertas": ...},
#     {"modelo": "XGBoost","AUC-PR": ..., "Recall": ..., "Precision": ..., "F1": ..., "n_alertas": ...},
# ])
# print(df_metrics.to_string(index=False))

# fig_aucpr: curvas PR superpuestas
# fig_aucpr.savefig(FIGURES_DIR / "aucpr_models.png", dpi=150, bbox_inches="tight")

print("§6 pendiente — Dev-2")

## §7 — Explicabilidad SHAP

**Owner:** Dev-3  
**Input:** `model_xgb`, `X_train`, `X_test`, `FEATURE_COLS`  
**Output:** `explainer`, `shap_values`, `fig_shap_bar`, `fig_shap_scatter`  
**Dependencias:** §5 (handoff H3)

In [ ]:
import shap

# TODO Dev-3:
# explainer   = shap.TreeExplainer(model_xgb)
# shap_values = explainer.shap_values(X_test)

# fig_shap_bar: importancia global
# shap.summary_plot(shap_values, X_test, plot_type="bar", show=False)
# fig_shap_bar = plt.gcf()
# fig_shap_bar.savefig(FIGURES_DIR / "shap_global_bar.png", dpi=150, bbox_inches="tight")

# fig_shap_scatter: beeswarm con dirección del efecto
# shap.summary_plot(shap_values, X_test, show=False)
# fig_shap_scatter = plt.gcf()
# fig_shap_scatter.savefig(FIGURES_DIR / "shap_beeswarm.png", dpi=150, bbox_inches="tight")

def clasifica_accion(shap_row: dict) -> str:
    """Retorna acción recomendada basada en el driver SHAP dominante del sitio."""
    # TODO Dev-3: implementar lógica
    # - throughput/capacidad crónico → "Inversión tecnológica"
    # - disponibilidad puntual AVA   → "Mantenimiento programado"
    # - sin señal clara              → "Monitoreo"
    raise NotImplementedError

print("§7 pendiente — Dev-3")

> **Hallazgo [completar]:** ...  
> **Implicación:** ...

## §8 — Segmentación de sitios

**Owner:** Dev-3  
**Input:** `df_weekly` (features agregadas por sitio)  
**Output:** `df_sites_clustered` (con columnas `cluster` y `perfil`)  
**Dependencias:** §2 (Parquet)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# TODO Dev-3:
# 1. Agregar df_weekly a nivel site_id (promedio del período)
# 2. StandardScaler sobre FEATURE_COLS
# 3. KMeans k=3..5, elegir k con silhouette score
# 4. Etiquetar clusters con perfil descriptivo
# 5. fig_cluster_scatter (PCA 2D o radar)

# df_sites_clustered esperado:
# site_id | avg_AVA | avg_DC | avg_THP | cluster | perfil

# fig_cluster_scatter.savefig(FIGURES_DIR / "cluster_sitios.png", dpi=150, bbox_inches="tight")

print("§8 pendiente — Dev-3")

## §9 — Ranking accionable (output final)

**Owner:** Dev-3  
**Input:** `df_weekly` (test), `y_prob_xgb`, `shap_values`, `df_sites_clustered`  
**Output:** `df_ranking` → `output/ranking_sitios.parquet`  
**Dependencias:** §5, §7, §8

In [ ]:
# TODO Dev-3: construir df_ranking con schema:
# site_id | score_riesgo | prioridad_sitio | kpis_driver |
# umbral_incumplido | accion_recomendada | cluster | semanas_en_riesgo

# df_ranking = df_test[[SITE_ID_COL]].copy()
# df_ranking["score_riesgo"] = y_prob_xgb
# df_ranking["prioridad_sitio"] = 0  # actualizar cuando cliente provea campo
# df_ranking["kpis_driver"] = ...    # top 2 features SHAP por sitio
# df_ranking["umbral_incumplido"] = ...
# df_ranking["accion_recomendada"] = df_ranking.apply(lambda r: clasifica_accion(...), axis=1)
# df_ranking = df_ranking.merge(df_sites_clustered[[SITE_ID_COL, "cluster"]], on=SITE_ID_COL, how="left")
# df_ranking["semanas_en_riesgo"] = ...
# df_ranking = df_ranking.sort_values("score_riesgo", ascending=False)
# df_ranking.to_parquet(RANKING_PATH, index=False)

# print(f"Ranking guardado: {RANKING_PATH}")
# print(f"Top 10 sitios de mayor riesgo:")
# display(df_ranking.head(10))

print("§9 pendiente — Dev-3")

---

## Checklist de entrega

- [ ] Kernel Restart → Run All completa sin error
- [ ] `output/modeling_dataset_weekly.parquet` generado
- [ ] `output/ranking_sitios.parquet` generado con 8 columnas del schema
- [ ] `df_metrics` impreso con AUC-PR para los 3 modelos
- [ ] Al menos 1 figura SHAP global en `output/figures/`
- [ ] `RANDOM_STATE = 42` en todos los modelos
- [ ] `TEST_CUTOFF_DATE` documentado con justificación
- [ ] Variables con nombres definidos en `entregables/pipeline.md`